In [ ]:
%pip install --pre torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/nightly/cpu
%pip install numpy
%pip install matplotlib
%pip install torchmetrics
%pip install fastfcgr
%pip install Bio

In [ ]:
import torch
import numpy as np
import random

#Setting seeds for reproducability
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.use_deterministic_algorithms(True)

In [ ]:
from amr.amr_utility import load_gene_data, create_gene_datasets
create_gene_datasets("./", "data/ds1/")

# Quickly Test Dataset
ds = load_gene_data("data/ds1", "Klebsiella_pneumoniae_aztreonam", "acrR")
print(ds["test"])
print(ds["train"])



In [ ]:
from amr.dataset import HybridGenomeDataset
from torch import device

def get_device() -> device:
    device_string = "cpu"
    if torch.cuda.is_available():
        device_string = "cuda"
    elif torch.mps.is_available():
        device_string = "mps"
    

    return torch.device(device_string)

In [ ]:
from torchmetrics.classification import BinaryConfusionMatrix
from torch import device
from torch import nn

from evaluation.epochmetrics import EpochMetrics
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

def step(metrics : EpochMetrics,dataloader, device : device, model : nn.Module, criterion, optimizer,val=False,test=False):    
    model.eval() if val else model.train()

    loss, correct, total = 0, 0, 0
    confmat = BinaryConfusionMatrix().to(device)
    
    t0 = 0
    t1 = 0
    
    class_1_labels = []
    class_1_probs = []
    
    for inputs, labels in dataloader:
        images = inputs[0].to(device)
        sequences = inputs[1].to(device)
        labels = labels.to(device)
        batch_size = labels.size(0)

        outputs = model((images, sequences))
        if test:
            class_1_probs.append(F.softmax(outputs, dim=1)[:, 0].cpu().numpy())
            class_1_labels.append(labels[:, 0].cpu())
            

            
        forward_loss = criterion(outputs, labels.float())
            
        if not val:
            optimizer.zero_grad()
            forward_loss.backward()
            optimizer.step()

        loss += forward_loss.item() * batch_size
        _, predicted = torch.max(outputs, 1)
        labels = torch.argmax(labels, dim=1)  
        
        num_zeros = (labels == 0).sum().item()
        num_ones = (labels == 1).sum().item()
        if not val:
            t0 += num_zeros
            t1 += num_ones
              
        confmat.update(predicted, labels)
        total += batch_size
        correct += (predicted == labels).sum().item()

    epoch_loss = loss / total
    epoch_acc = correct / total
    matrix = confmat.compute()
    
    if(val):
        metrics.val_confusion = matrix.tolist()
        metrics.val_acc = epoch_acc
        metrics.val_loss = epoch_loss
    else:
        metrics.train_confusion = matrix.tolist()
        metrics.train_acc = epoch_acc
        metrics.train_loss = epoch_loss
        
    if len(class_1_probs) == 0:
        class_1_probs.append(np.ndarray([0]))
    if len(class_1_labels) == 0:
        class_1_labels.append(np.ndarray([0]))
    return t0,t1,np.concatenate(class_1_probs, axis=0),np.concatenate(class_1_labels, axis=0)

In [ ]:
from typing import List
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from amr.dataloaders import get_train_val_dataloaders
from configs.config import GeneModelConfig
from evaluation.epochmetrics import EpochMetrics


class WeightSaver:
    def __init__(self, model):
        self.model = model
        self.best_loss = float("inf")
        self.best_acc = 0
        self.best_weights = None

    def check_and_save(self, metrics: EpochMetrics):
        if (metrics.val_loss < self.best_loss) or (
            metrics.val_loss == self.best_loss and metrics.val_acc > self.best_acc
        ):
            self.best_loss = metrics.val_loss
            self.best_acc = metrics.val_acc
            self.best_weights = self.model.state_dict()

    def restore_best_weights(self):
        self.model.load_state_dict(self.best_weights)


def train(model, device, optimizer, config: GeneModelConfig):
    train_loader, val_loader, weights = get_train_val_dataloaders(config=config)

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=30,
        T_mult=1,
        eta_min=1e-10,
    )

    class_weights = torch.tensor(
        [weights[1], weights[0]] if config.lossweighting else [1.0, 1.0]
    ).to(device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    weight_saver = WeightSaver(model=model)

    progress: List[EpochMetrics] = []
    num_epochs = config.epochs

    t0 = 0
    t1 = 0

    for epoch in range(num_epochs):
        metrics = EpochMetrics(epoch=epoch)

        stept0, stept1, probs, labels = step(
            metrics, train_loader, device, model, criterion, optimizer
        )
        t0 += stept0
        t1 += stept1
        with torch.no_grad():
            stept0, stept1, probs, labels = step(
                metrics, val_loader, device, model, criterion, optimizer, True
            )

        scheduler.step(metrics.val_loss)
        progress.append(metrics)
        weight_saver.check_and_save(metrics)
        metrics.print_train(num_epochs)

    print("Class Counts:")
    print(t0)
    print(t1)

    if config.returnlowestvalloss:
        weight_saver.restore_best_weights()

    return model, progress, class_weights

In [ ]:
from amr.dataloaders import get_dataloader, get_test_dataloader


def test(
    model : nn.Module, device : device, optimizer, config: GeneModelConfig, class_weights
) -> List[EpochMetrics]:
    results : List[EpochMetrics] = []
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    for gene in config.genes:
        print("Gene: ", gene)

        test_loader = get_test_dataloader(config=config)
        
        metrics = EpochMetrics()
        with torch.no_grad():
            t0,t1,class_probs,class_labels = step(metrics, test_loader, device, model, criterion, optimizer, True,test=True)
            fpr, tpr, thresholds = roc_curve(class_labels, class_probs)
            metrics.test_roc_fpr = fpr
            metrics.test_roc_tpr = tpr
            metrics.print_val()
            results.append(metrics)
    return results

In [ ]:

from net.HybridGenomeNet import HybridGenomeNet
from net.OnlyFCGRNet import OnlyFCGRNet
from net.OnlySequenceNet import OnlySequenceNet
import matplotlib.pyplot as plt
from evaluation.trainingresults import TrainingResults
from configs.config import GeneModelConfig
from torch.optim.optimizer import Optimizer
import os



def package_results(progress : List[EpochMetrics], test_results : List[EpochMetrics], gene, config) -> TrainingResults:
    package = TrainingResults(
        gene=gene,
        test_loss=test_results[0].val_loss,
        test_acc=test_results[0].val_acc,
        confusion_matrix=test_results[0].val_confusion,
        config=config,
        test_roc_fpr=test_results[0].test_roc_fpr,
        test_roc_tpr=test_results[0].test_roc_tpr
    )

    for item in progress:
        package.accs.append(item.train_acc)
        package.val_accs.append(item.val_acc)
        package.tpr.append(item.train_tpr)
        package.fpr.append(item.train_fpr)
        package.losses.append(item.train_loss)
        package.val_losses.append(item.val_loss)
        package.val_tpr.append(item.val_tpr)
        package.val_fpr.append(item.val_fpr)

    return package

def get_model(config : GeneModelConfig) -> nn.Module:
    model = None
    if config.onlyfcgr:
        model = OnlyFCGRNet(dropout=config.dropout, k=config.k)
    elif config.onlysequence:
        model = OnlySequenceNet(dropout=config.dropout,k=config.k)
    else:
        model = HybridGenomeNet(dropout=config.dropout, k=config.k)
    return model

def train_and_test_config(config: GeneModelConfig) -> List[TrainingResults]:
    full_results : List[TrainingResults] = []
    device = get_device()
    for gene in config.genes:
        model = get_model(config).to(device)

        optimizer: Optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
        )

        model, progress, class_weights = train(model, device, optimizer, config)
        model_folder = "models/" + config.pathogen
        model_path = model_folder + "/" + gene + ".pth"
        if not os.path.exists(model_folder):
            os.makedirs(model_folder)
        torch.save(model.state_dict(), model_path)
        del model
        
        test_model = get_model(config).to(device)
        test_model.load_state_dict(torch.load(model_path))
        
        test_results = test(test_model, device, optimizer, config, class_weights)
        full_results.append(package_results(progress, test_results, gene, config))
        
    return full_results

In [ ]:
from typing import List
import os

def insert_single_plot(
    slot, x, y_training, y_validation, x_label, y_label, final, gene
):
    step = 20  # Your x-axis step size
    slot.plot(x, y_training, marker="o", label=f"Training {y_label}")
    slot.plot(x, y_validation, marker="x", label=f"Validation {y_label}")
    slot.set_title(f"{y_label} - {gene} - Final Test: {final}")
    slot.set_xlabel(x_label)
    slot.set_ylabel(y_label)
    slot.set_xticks(range(0, len(y_training) + 1, step))
    slot.legend()
    slot.grid(True)


def plot_roc_curve(ax_roc,fpr, tpr, roc_auc,gene):
    ax_roc.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC curve (AUC = {roc_auc:.2f})')
    ax_roc.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random baseline')
    ax_roc.set_xlim(0.0, 1.0)
    ax_roc.set_ylim(0.0, 1.05)
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title(f'Test Receiver Operating Characteristic (ROC) Curve for {gene}')
    ax_roc.legend(loc="lower right")
    ax_roc.grid(True)

def format_hyperparams(config: GeneModelConfig) -> str:

    rows = [
        f"Data Settings:            K-Mers: {config.k}    | Train/Val Split: {config.trainvalsplit}",
        f"Training Settings:       Batch Size: {config.batch_size} | Learning Rate: {config.learning_rate} | Epochs: {config.epochs}",
        f"Regularization:           Noise: {config.noise} | Weight Decay: {config.weight_decay} | Dropout: {config.dropout}",
        f"Imbalance Handling:  Loss Weighting: {config.lossweighting} | Rare Class Sampling: {config.rareclasssampling} | Sample Reuse: {config.rareclasssamplerreplacement}",
        f"Model Settings:          Return Lowest Val Loss: {config.returnlowestvalloss} | Only FCGR: {config.onlyfcgr}"
    ]
    return "\n".join(rows)


def plot_full_results(full_results: List[TrainingResults],savefolder : str,config : GeneModelConfig):
    genes = [result.gene for result in full_results]
    n = len(genes)

    # 2 rows per gene: row 0 = acc/loss, row 1 = roc/text
    fig, axes = plt.subplots(
        nrows=n, ncols=3, figsize=(21, 5 * n)
    )  # 2 columns: acc/loss and roc/text

    if n == 1:
        axes = [axes]  # Handle single gene case

    for i, result in enumerate(full_results):
        gene = result.gene
        epochs = range(1, result.config.epochs + 1)

        # Accuracy
        ax_acc = axes[i][0]
        insert_single_plot(
            ax_acc,
            epochs,
            result.accs,
            result.val_accs,
            "Epoch",
            "Accuracy",
            round(result.test_acc, 2),
            gene,
        )
        ax_acc.set_ylim(0, 1)

        # Loss
        ax_loss = axes[i][1]
        insert_single_plot(
            ax_loss,
            epochs,
            result.losses,
            result.val_losses,
            "Epoch",
            "Loss",
            round(result.test_loss, 2),
            gene,
        )

        ax_loss.set_ylim(0,2)

        # ROC Curve
        ax_roc = axes[i][2]
        fprs = result.test_roc_fpr
        tprs = result.test_roc_tpr
        roc_auc = auc(fprs, tprs)
        plot_roc_curve(
            ax_roc,
            fprs,
            tprs,
            roc_auc,
            gene
        )

    plt.suptitle(format_hyperparams(config), x=0.33,y=0.99, ha="left", va="top")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    save_path = os.path.join(savefolder, f"results.svg")
    fig.savefig(save_path, bbox_inches="tight")
    plt.close(fig)    

In [ ]:
import pandas as pd


def save_results(full_results : List[TrainingResults], config : GeneModelConfig,config_dir : str):
    for result in full_results:
        gene = result.gene

        # Prepare a DataFrame for epoch-wise metrics
        df = pd.DataFrame(
            {
                "epoch": range(1, config.epochs + 1),
                "train_accuracy": result.accs,
                "val_accuracy": result.val_accs,
                "train_loss": result.losses,
                "val_loss": result.val_losses,
            }
        )
        df.to_csv(f"{config_dir}/{gene}_training_results.csv", index=False)

        # Save test results (single-row DataFrame)
        test_df = pd.DataFrame(
            {"test_loss": [result.test_loss], "test_accuracy": [result.test_acc]}
        )
        test_df.to_csv(f"{config_dir}/{gene}_test_results.csv", index=False)

        # Optionally, save confusion matrix if it’s a 2D array
        cm = result.confusion_matrix
        cm_df = pd.DataFrame(cm)
        cm_df.to_csv(f"{config_dir}/{gene}_confusion_matrix.csv", index=False)

# How To Use
Choose the configs you want to train from the given imports in the cell below.
All available configs are in the configs folder.
Configs can also be used to change which model-variation to train (Hybrid, FCGR-Only, Sequence-only)


In [ ]:
from configs.staphy_config_fcgr import (
    staphy_configs,
    fusA_config,
    dfrB_config,
    pbp4_config,
    grlA_config,
    grlB_config,
    gyrA_config,
    ileS_config,
    pbp2_config,
    pbp4_promoter_config,
    rpoB_config
)
from datetime import datetime
import os
import json
from dataclasses import asdict


def save_config_to_json(config: GeneModelConfig, path: str):
    with open(path, "w") as f:
        json.dump(asdict(config), f, indent=4)


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = os.path.join("results", timestamp)
os.makedirs(results_dir, exist_ok=True)


for config in staphy_configs:
    torch.manual_seed(0)
    np.random.seed(0)
    random.seed(0)
    torch.use_deterministic_algorithms(True)
    config_dir = os.path.join(results_dir, config.name)
    os.makedirs(config_dir, exist_ok=True)
    results = train_and_test_config(config)
    plot_full_results(results, config_dir, config)
    save_config_to_json(config, os.path.join(config_dir, "config.json"))
    save_results(results, config, config_dir)